# Data Farming with Categorical (Enum) Factors

This notebook demonstrates how to run a data farming experiment when one of the factors is a categorical variable, specifically an enumeration (`Enum`) type.

The `simopt.data_farming.create_design` function expects numerical values for all factors. To work around this for categorical factors, we can:
1.  **Map the Enum to Integers**: Create a mapping from each enum member to a unique integer.
2.  **Generate the Design**: Use these integers to define the levels of the factor in the experimental design.
3.  **Define an Evaluation Function**: Create a wrapper function that takes a design point, looks up the original enum member from the integer, and then runs the simulation.
4.  **Run the Experiment**: Execute the data farming run.
5.  **Post-process Results**: Map the integer values in the results back to the original enum member names for easier analysis.

We will demonstrate this by evaluating the performance of the `ASTROMoRF` solver with different `PolyBasisType` options on the `SSCONT` problem.

## 1. Import Required Libraries

In [15]:
import sys
from pathlib import Path

# Take the current directory, find the parent, and add it to the system path
sys.path.append(str(Path.cwd().parent))

In [16]:

import pandas as pd

from simopt.experiment_base import ProblemsSolvers
from simopt.solvers.astromorf import PolyBasisType

## 2. Create Enum-to-Integer Mapping

In [17]:
# Create a list of the enum members
poly_basis_options = list(PolyBasisType)

# Create a mapping from the enum member to an integer
poly_basis_to_int = {basis: i for i, basis in enumerate(poly_basis_options)}

# Create a reverse mapping from the integer to the enum member
int_to_poly_basis = {i: basis for i, basis in enumerate(poly_basis_options)}

print("Enum to Integer Mapping:")
print(poly_basis_to_int)
print("\nInteger to Enum Mapping:")
print(int_to_poly_basis)

Enum to Integer Mapping:
{<PolyBasisType.HERMITE: 'hermite'>: 0, <PolyBasisType.LEGENDRE: 'legendre'>: 1, <PolyBasisType.CHEBYSHEV: 'chebyshev'>: 2, <PolyBasisType.MONOMIAL: 'monomial'>: 3, <PolyBasisType.NATURAL: 'natural'>: 4, <PolyBasisType.MONOMIAL_POLY: 'monomial_polynomial'>: 5, <PolyBasisType.LAGRANGE: 'lagrange'>: 6, <PolyBasisType.NFP: 'nfp'>: 7, <PolyBasisType.LAGUERRE: 'laguerre'>: 8}

Integer to Enum Mapping:
{0: <PolyBasisType.HERMITE: 'hermite'>, 1: <PolyBasisType.LEGENDRE: 'legendre'>, 2: <PolyBasisType.CHEBYSHEV: 'chebyshev'>, 3: <PolyBasisType.MONOMIAL: 'monomial'>, 4: <PolyBasisType.NATURAL: 'natural'>, 5: <PolyBasisType.MONOMIAL_POLY: 'monomial_polynomial'>, 6: <PolyBasisType.LAGRANGE: 'lagrange'>, 7: <PolyBasisType.NFP: 'nfp'>, 8: <PolyBasisType.LAGUERRE: 'laguerre'>}


## 3. Generate the Experimental Design

In [ ]:
# Define the factors for the experiment.
# The `polynomial_basis` factor uses the integer representations.
factor_settings = {
    "polynomial_basis": list(int_to_poly_basis.keys())
}

# Generate the design using a full factorial design type.
from simopt.experiment_base import create_design
design = create_design(factor_settings, design_type="full_factorial")

print("Experimental Design:")
display(design)

TypeError: create_design() missing 2 required positional arguments: 'factor_headers' and 'factor_settings'

: 

## 4. Define the Evaluation Function

In [ ]:
from simopt.experiment_base import ProblemSolver
# Create a list of ProblemSolver objects for the experiment
experiments = []
for basis_int, basis_enum in int_to_poly_basis.items():
    experiments.append(
        ProblemSolver(
            solver_name="ASTROMORF",
            problem_name="SSCONT-1",
            solver_fixed_factors={"polynomial_basis": basis_enum},
            problem_fixed_factors={"budget": 200},
        )
    )

## 5. Run the Datafarming Experiment

In [ ]:
# Create ProblemsSolvers experiment
experiment = ProblemsSolvers(experiments=experiments)

# check compatibility of selected solvers and problems
experiment.check_compatibility()

## 6. Analyze the Results

In [ ]:
# For easier analysis, you can inspect the results dataframes

# Display the recommended solutions
print("Recommended Solutions:")
display(recommended_solns)

# Display all responses
print("All Responses:")
display(all_responses)

# You can group by solver (which corresponds to the polynomial basis) to analyze performance
print("\nAverage Function Evaluations per Basis Type:")
avg_fn_evals = all_responses.groupby('solver')['response_value'].mean().sort_values()
display(avg_fn_evals)

In [ ]:
# Run experiment
experiment.run(n_macroreps=3)

# Post-processing
# View results in a DataFrame
recommended_solns = experiment.get_recommended_solutions()
# Get a tidy dataframe of all responses
all_responses = experiment.get_all_responses()

# Get a tidy dataframe of all post-replications
all_postreps = experiment.get_all_postreplications()

# Get a tidy dataframe of all post-replications at x0 and x*
all_postreps_init_opt = experiment.get_all_postreplications_init_opt()